# Diseño de Esquemas

Normalización (1NF–3NF), claves foráneas, ON DELETE CASCADE, patrones de diseño

## Introducción

Un buen diseño de base de datos previene redundancia, inconsistencias y problemas de escalabilidad desde el inicio. La normalización es el proceso de organizar tablas y columnas para reducir duplicación y dependencias problemáticas. Las formas normales (1NF, 2NF, 3NF) son guías progresivas hacia un diseño limpio. Entender relaciones (1:1, 1:N, N:M) y claves foráneas es fundamental para modelar cualquier dominio del mundo real.

### Objetivos de Aprendizaje

- Aplicar 1NF, 2NF y 3NF para eliminar redundancia y dependencias problemáticas
- Diseñar relaciones 1:1, 1:N y N:M con claves primarias y foráneas
- Implementar tablas de unión (junction tables) para relaciones N:M
- Usar constraints (UNIQUE, NOT NULL, CHECK, FOREIGN KEY) para integridad de datos
- Modelar esquemas reales: e-commerce, blog, sistema de cursos

## 1NF — Valores atómicos y sin repetición

> Primera Forma Normal: cada celda debe contener un único valor atómico (indivisible), y cada fila debe ser única. Viola 1NF: guardar múltiples valores en una columna ("Ana,Luis,Sara"), o tener grupos repetidos de columnas (telefono1, telefono2, telefono3). La solución es siempre crear una tabla separada para los valores múltiples.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE productos_mal (
    id INTEGER PRIMARY KEY,
    nombre TEXT,
    categorias TEXT
)''')
cursor.executemany('INSERT INTO productos_mal VALUES (?,?,?)', [
    (1, 'iPhone 15', 'Tech,Gadgets,Móvil'),
    (2, 'Laptop Pro', 'Tech,Computadoras'),
])

cursor.execute('''CREATE TABLE productos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL
)''')
cursor.execute('''CREATE TABLE categorias (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT UNIQUE NOT NULL
)''')
cursor.execute('''CREATE TABLE producto_categoria (
    producto_id INTEGER REFERENCES productos(id),
    categoria_id INTEGER REFERENCES categorias(id),
    PRIMARY KEY (producto_id, categoria_id)
)''')

cursor.executemany('INSERT INTO productos (nombre) VALUES (?)', [('iPhone 15',), ('Laptop Pro',)])
cursor.executemany('INSERT INTO categorias (nombre) VALUES (?)', [
    ('Tech',), ('Gadgets',), ('Móvil',), ('Computadoras',)
])
cursor.executemany('INSERT INTO producto_categoria VALUES (?,?)', [
    (1,1),(1,2),(1,3), (2,1),(2,4)
])
conn.commit()

cursor.execute('''
    SELECT p.nombre, GROUP_CONCAT(c.nombre, ', ') AS categorias
    FROM productos p
    JOIN producto_categoria pc ON p.id = pc.producto_id
    JOIN categorias c ON pc.categoria_id = c.id
    GROUP BY p.nombre
''')
print("── Productos con categorías (1NF) ──")
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[1]}")

conn.close()

## 2NF — Sin dependencias parciales

> Segunda Forma Normal (requiere 1NF): cada columna no-clave debe depender de TODA la clave primaria, no de una parte. Problema típico: tabla con clave compuesta (pedido_id, producto_id) donde el nombre del producto depende solo de producto_id, no de la clave completa. Solución: separar en tablas por entidad.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE pedido_detalle_mal (
    pedido_id INTEGER,
    producto_id INTEGER,
    nombre_producto TEXT,
    precio_producto REAL,
    cantidad INTEGER,
    PRIMARY KEY (pedido_id, producto_id)
)''')

cursor.executescript('''
CREATE TABLE pedidos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    cliente TEXT NOT NULL,
    fecha TEXT DEFAULT CURRENT_TIMESTAMP
);
CREATE TABLE productos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    precio REAL NOT NULL
);
CREATE TABLE pedido_items (
    pedido_id INTEGER REFERENCES pedidos(id),
    producto_id INTEGER REFERENCES productos(id),
    cantidad INTEGER NOT NULL CHECK(cantidad > 0),
    PRIMARY KEY (pedido_id, producto_id)
);

INSERT INTO pedidos (cliente) VALUES ('Ana García'), ('Luis Pérez');
INSERT INTO productos (nombre, precio) VALUES ('Laptop',1200), ('Mouse',25), ('Monitor',400);
INSERT INTO pedido_items VALUES (1,1,1), (1,2,2), (2,3,1), (2,1,1);
''')
conn.commit()

cursor.execute('''
    SELECT p.cliente, GROUP_CONCAT(pr.nombre || ' x' || pi.cantidad) AS items,
           SUM(pr.precio * pi.cantidad) AS total
    FROM pedidos p
    JOIN pedido_items pi ON p.id = pi.pedido_id
    JOIN productos pr ON pi.producto_id = pr.id
    GROUP BY p.id
''')
print("── Pedidos con total (2NF) ──")
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[1]} → ${row[2]:,.2f}")

conn.close()

## 3NF — Sin dependencias transitivas

> Tercera Forma Normal (requiere 2NF): ninguna columna no-clave debe depender de otra columna no-clave. El problema clásico: tabla empleados donde departamento_nombre depende de departamento_id (no del empleado directamente). Si el nombre del departamento cambia, hay que actualizar miles de filas. Solución: tabla de departamentos separada.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE empleados_mal (
    id INTEGER PRIMARY KEY,
    nombre TEXT,
    depto_id INTEGER,
    depto_nombre TEXT,
    depto_ubicacion TEXT
)''')

cursor.executescript('''
CREATE TABLE departamentos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT UNIQUE NOT NULL,
    ubicacion TEXT,
    presupuesto REAL
);
CREATE TABLE empleados (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    email TEXT UNIQUE,
    salario REAL,
    depto_id INTEGER REFERENCES departamentos(id)
);

INSERT INTO departamentos (nombre, ubicacion, presupuesto) VALUES
    ('Ingeniería', 'Piso 3', 500000),
    ('Marketing', 'Piso 1', 200000),
    ('Ventas', 'Piso 2', 300000);

INSERT INTO empleados (nombre, email, salario, depto_id) VALUES
    ('Ana García', 'ana@co.com', 85000, 1),
    ('Luis Pérez', 'luis@co.com', 62000, 2),
    ('Sara López', 'sara@co.com', 91000, 1),
    ('Pedro Ruiz', 'pedro@co.com', 55000, 3);
''')
conn.commit()

cursor.execute("UPDATE departamentos SET nombre = 'Engineering' WHERE nombre = 'Ingeniería'")
conn.commit()

cursor.execute('''
    SELECT e.nombre, d.nombre AS departamento, d.ubicacion, e.salario
    FROM empleados e JOIN departamentos d ON e.depto_id = d.id
    ORDER BY d.nombre, e.salario DESC
''')
print("── Empleados (3NF) ──")
for row in cursor.fetchall():
    print(f"  {row[0]:10} | {row[1]:12} | {row[2]:7} | ${row[3]:,.0f}")

conn.close()

## Relaciones 1:N y N:M con claves foráneas

> Las relaciones definen cómo se conectan las entidades. 1:N (uno a muchos): un cliente tiene muchos pedidos — la FK va en la tabla "muchos" (pedidos.cliente_id). N:M (muchos a muchos): un estudiante puede estar en muchos cursos y un curso tiene muchos estudiantes — requiere tabla de unión intermedia.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.execute('PRAGMA foreign_keys = ON')

cursor.executescript('''
CREATE TABLE autores (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    pais TEXT
);
CREATE TABLE libros (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    titulo TEXT NOT NULL,
    anio INTEGER,
    autor_id INTEGER NOT NULL REFERENCES autores(id) ON DELETE RESTRICT
);
CREATE TABLE generos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT UNIQUE NOT NULL
);
CREATE TABLE libro_genero (
    libro_id INTEGER REFERENCES libros(id) ON DELETE CASCADE,
    genero_id INTEGER REFERENCES generos(id),
    PRIMARY KEY (libro_id, genero_id)
);

INSERT INTO autores (nombre, pais) VALUES
    ('Gabriel García Márquez','Colombia'),('George Orwell','UK'),('Yuval Noah Harari','Israel');
INSERT INTO libros (titulo, anio, autor_id) VALUES
    ('Cien años de soledad',1967,1),('El amor en los tiempos del cólera',1985,1),
    ('1984',1949,2),('Rebelión en la granja',1945,2),('Sapiens',2011,3);
INSERT INTO generos (nombre) VALUES ('Ficción'),('Realismo mágico'),('Distopía'),('No ficción'),('Historia');
INSERT INTO libro_genero VALUES
    (1,1),(1,2),(2,1),(2,2),(3,1),(3,3),(4,1),(4,3),(5,4),(5,5);
''')
conn.commit()

print("── Libros por autor (1:N) ──")
cursor.execute('''
    SELECT a.nombre, COUNT(l.id) AS libros
    FROM autores a LEFT JOIN libros l ON a.id = l.autor_id
    GROUP BY a.nombre
''')
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[1]} libro(s)")

print("\n── Libros con géneros (N:M) ──")
cursor.execute('''
    SELECT l.titulo, GROUP_CONCAT(g.nombre, ' + ') AS generos
    FROM libros l
    JOIN libro_genero lg ON l.id = lg.libro_id
    JOIN generos g ON lg.genero_id = g.id
    GROUP BY l.titulo ORDER BY l.titulo
''')
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[1]}")

conn.close()

## Constraints — Integridad garantizada

> Los constraints son reglas que la base de datos enforcea automáticamente. PRIMARY KEY (único + no nulo), UNIQUE (sin duplicados), NOT NULL (obligatorio), CHECK (condición custom), FOREIGN KEY (referencial). Son la primera línea de defensa contra datos corruptos — mejor que validar solo en código de aplicación.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.execute('PRAGMA foreign_keys = ON')

cursor.execute('''CREATE TABLE usuarios (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    email TEXT UNIQUE NOT NULL,
    nombre TEXT NOT NULL,
    edad INTEGER CHECK(edad >= 18 AND edad <= 120),
    rol TEXT NOT NULL DEFAULT 'usuario' CHECK(rol IN ('admin','editor','usuario')),
    creditos REAL DEFAULT 0.0 CHECK(creditos >= 0),
    creado_en TEXT DEFAULT CURRENT_TIMESTAMP
)''')

cursor.execute('''CREATE TABLE transacciones (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    usuario_id INTEGER NOT NULL REFERENCES usuarios(id) ON DELETE CASCADE,
    monto REAL NOT NULL CHECK(monto > 0),
    tipo TEXT NOT NULL CHECK(tipo IN ('deposito','retiro')),
    fecha TEXT DEFAULT CURRENT_TIMESTAMP
)''')

cursor.execute("INSERT INTO usuarios (email, nombre, edad, rol) VALUES (?,?,?,?)",
               ('ana@co.com', 'Ana García', 28, 'admin'))
conn.commit()
print("✓ Usuario válido insertado")

tests = [
    ("Email duplicado",
     "INSERT INTO usuarios (email, nombre, edad) VALUES ('ana@co.com','Luis',25)"),
    ("Edad fuera de rango",
     "INSERT INTO usuarios (email, nombre, edad) VALUES ('luis@co.com','Luis',15)"),
    ("Rol inválido",
     "INSERT INTO usuarios (email, nombre, edad, rol) VALUES ('s@co.com','Sara',30,'superadmin')"),
    ("Créditos negativos",
     "UPDATE usuarios SET creditos = -100 WHERE email = 'ana@co.com'"),
]
for nombre, sql in tests:
    try:
        cursor.execute(sql)
        print(f"✗ {nombre}: debería haber fallado")
    except Exception as e:
        print(f"✓ {nombre}: bloqueado → {str(e)[:50]}")

conn.close()

## Schema completo: E-commerce

Diseña el esquema normalizado de una tienda online con usuarios, productos, pedidos y categorías.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.execute('PRAGMA foreign_keys = ON')

cursor.executescript('''
-- Entidades base (sin FKs)
CREATE TABLE categorias (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT UNIQUE NOT NULL,
    descripcion TEXT
);
CREATE TABLE usuarios (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    email TEXT UNIQUE NOT NULL,
    nombre TEXT NOT NULL,
    ciudad TEXT,
    creado_en TEXT DEFAULT CURRENT_TIMESTAMP
);

-- Entidades dependientes
CREATE TABLE productos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    precio REAL NOT NULL CHECK(precio > 0),
    stock INTEGER DEFAULT 0 CHECK(stock >= 0),
    categoria_id INTEGER REFERENCES categorias(id)
);
CREATE TABLE pedidos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    usuario_id INTEGER NOT NULL REFERENCES usuarios(id),
    estado TEXT DEFAULT 'pendiente' CHECK(estado IN ('pendiente','enviado','entregado','cancelado')),
    total REAL DEFAULT 0,
    fecha TEXT DEFAULT CURRENT_TIMESTAMP
);
CREATE TABLE pedido_items (
    pedido_id INTEGER REFERENCES pedidos(id) ON DELETE CASCADE,
    producto_id INTEGER REFERENCES productos(id),
    cantidad INTEGER NOT NULL CHECK(cantidad > 0),
    precio_unit REAL NOT NULL,
    PRIMARY KEY (pedido_id, producto_id)
);

-- Datos de prueba
INSERT INTO categorias (nombre) VALUES ('Tech'),('Oficina'),('Hogar');
INSERT INTO usuarios (email, nombre, ciudad) VALUES
    ('ana@co.com','Ana García','CDMX'),('luis@co.com','Luis Pérez','GDL');
INSERT INTO productos (nombre, precio, stock, categoria_id) VALUES
    ('Laptop Pro',1299.99,10,1),('Mouse BT',29.99,50,1),
    ('Silla Ergonómica',349.99,15,2),('Monitor 4K',499.99,8,1);
INSERT INTO pedidos (usuario_id, estado) VALUES (1,'enviado'),(2,'pendiente');
INSERT INTO pedido_items VALUES
    (1,1,1,1299.99),(1,2,2,29.99),(2,3,1,349.99),(2,4,1,499.99);
UPDATE pedidos SET total = (
    SELECT SUM(cantidad * precio_unit) FROM pedido_items WHERE pedido_id = pedidos.id
);
''')
conn.commit()

cursor.execute('''
    SELECT c.nombre AS categoria, COUNT(DISTINCT pi.pedido_id) AS pedidos,
           SUM(pi.cantidad) AS unidades, SUM(pi.cantidad * pi.precio_unit) AS ingresos
    FROM pedido_items pi
    JOIN productos p ON pi.producto_id = p.id
    JOIN categorias c ON p.categoria_id = c.id
    GROUP BY c.nombre ORDER BY ingresos DESC
''')
print("── Ventas por categoría ──")
for row in cursor.fetchall():
    print(f"  {row[0]:8}: {row[1]} pedidos, {row[2]} unidades, ${row[3]:,.2f}")

cursor.execute('''
    SELECT p.nombre, SUM(pi.cantidad) AS vendidos,
           SUM(pi.cantidad * pi.precio_unit) AS revenue
    FROM pedido_items pi JOIN productos p ON pi.producto_id = p.id
    GROUP BY p.nombre ORDER BY revenue DESC
''')
print("\n── Top productos ──")
for row in cursor.fetchall():
    print(f"  {row[0]:18}: {row[1]} vendidos, ${row[2]:,.2f}")

conn.close()

## Tips y Mejores Prácticas

> Diseña el esquema en papel (o en dbdiagram.io) antes de escribir CREATE TABLE. Identifica las entidades, sus atributos y las relaciones entre ellas. Un esquema bien pensado evita migraciones costosas después.

> ON DELETE CASCADE borra automáticamente los registros relacionados cuando se elimina el padre. Es conveniente pero peligroso: un DELETE en usuarios borra todos sus pedidos. Usa ON DELETE RESTRICT si prefieres que el DB impida el borrado cuando hay hijos — te obliga a limpiar manualmente.

> La desnormalización a veces es correcta. En tablas de reportes o data warehouses, guardar datos redundantes puede mejorar dramáticamente el rendimiento de queries analíticos. La regla es: normaliza para OLTP (operaciones), desnormaliza para OLAP (análisis).

> Guarda el precio en pedido_items al momento de la compra, no como FK a productos.precio. Si el precio del producto cambia mañana, los pedidos históricos deben mantener el precio original. Este es el antipatrón más común en diseños de e-commerce.

## Errores Comunes

### Guardar listas en una columna como strings separados por coma

¿Por qué ocurre?
- Viola 1NF y hace imposible filtrar, ordenar o contar por elemento individual. "Tech,Gadgets,Móvil" no se puede indexar ni filtrar eficientemente.

Solución
- Crea una tabla de relación separada. Para producto-categorías: tabla producto_categoria con (producto_id, categoria_id). Consulta con JOIN + GROUP_CONCAT si necesitas mostrarlas juntas.

### No activar PRAGMA foreign_keys = ON en SQLite

¿Por qué ocurre?
- SQLite NO enforcea foreign keys por defecto por compatibilidad histórica. Sin este pragma, puedes insertar referencias a IDs inexistentes sin error.

Solución
- Ejecuta cursor.execute("PRAGMA foreign_keys = ON") inmediatamente después de cada conexión nueva. En SQLAlchemy: create_engine(url, connect_args={"check_same_thread": False}).

### Usar el precio actual del producto en pedidos históricos

¿Por qué ocurre?
- Si guardas solo el producto_id en pedido_items y el precio cambia, todos los pedidos históricos mostrarán el precio nuevo — incorrecto para facturación.

Solución
- Guarda precio_unit en la tabla de items al momento de la compra: INSERT INTO pedido_items (pedido_id, producto_id, cantidad, precio_unit) VALUES (?, ?, ?, ?).

### Claves primarias no naturales vs naturales

¿Por qué ocurre?
- Usar email o DNI como PK parece conveniente pero es problemático: los emails cambian, los DNIs tienen formatos distintos por país, y son más lentos en JOINs que enteros.

Solución
- Usa siempre una PK artificial (INTEGER AUTOINCREMENT) como clave técnica. Agrega un UNIQUE constraint en el email/DNI para mantener unicidad sin usarlos como PK.